# 02: PSPNET SEGMENTATION

This notebook performs semantic segmentation on GSV images using PSPNet with ADE20K classes. The model segments each image into 150 semantic categories.

## MODULE SETUP

In [ ]:
# Mount Google Drive.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

# Set working directory to project folder.
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/hot_hem"
os.chdir(BASE_DIR)

print(f"Working directory: {BASE_DIR}")

In [ ]:
# Install segmentation models.
!pip install segmentation-models-pytorch -q

## IMPORT SETUP

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm import tqdm

import torch
import torchvision.transforms as T
from torchvision.models.segmentation import deeplabv3_resnet101

## PATH CONFIGURATION

In [ ]:
# Input paths.
IMAGE_DIR = Path("data/processing/images")
META_CSV = Path("data/processing/gsv/metadata.csv")

# Output paths.
CHECKPOINT_FILE = Path("data/processing/gsv/segmentation_checkpoint.json")

print("PATH CONFIGURATION")
print(f"Image directory: {IMAGE_DIR}")
print(f"Metadata file: {META_CSV}")
print(f"Checkpoint file: {CHECKPOINT_FILE}")

## LOAD METADATA

In [ ]:
# Load GSV metadata.
df = pd.read_csv(META_CSV)

print(f"Loaded {len(df)} GSV image records.")
print(f"Columns: {list(df.columns)}")
df.head()

## LOAD CHECKPOINT

In [ ]:
# Load checkpoint if exists.
if CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE, "r") as f:
        checkpoint = json.load(f)
        processed_uids = set(checkpoint.get("processed_uids", []))
    print(f"Loaded checkpoint with {len(processed_uids)} processed UIDs.")
else:
    processed_uids = set()
    print("No checkpoint found. Starting fresh.")

## LOAD SEGMENTATION MODEL

In [ ]:
# Set device.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load pre-trained DeepLabV3 model.
# Note: Using DeepLabV3 as proxy. For PSPNet with ADE20K, use appropriate model.
model = deeplabv3_resnet101(pretrained = True)
model = model.to(device)
model.eval()

print("Model loaded successfully.")

In [ ]:
# Define image preprocessing transforms.
preprocess = T.Compose([
    T.ToTensor(),
    T.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
    )
])

## SEGMENTATION HELPER FUNCTION

In [ ]:
def segment_image(image_path):
    """
    Perform semantic segmentation on input image.
    Returns segmentation mask with class IDs per pixel.
    """
    # Load image.
    img = Image.open(image_path).convert("RGB")
    original_size = img.size
    
    # Preprocess.
    input_tensor = preprocess(img)
    input_batch = input_tensor.unsqueeze(0).to(device)
    
    # Run inference.
    with torch.no_grad():
        output = model(input_batch)["out"][0]
    
    # Get class predictions.
    output_predictions = output.argmax(0).cpu().numpy()
    
    # Resize mask to original image size.
    mask_resized = cv2.resize(
        output_predictions.astype(np.uint8),
        original_size,
        interpolation = cv2.INTER_NEAREST
    )
    
    return mask_resized

## RUN SEGMENTATION

In [ ]:
# Process each image.
results = []
batch_size = 100

for idx, row in tqdm(df.iterrows(), total = len(df), desc = "Segmenting images"):
    uid = row["uid"]
    
    # Skip if already processed.
    if uid in processed_uids:
        continue
    
    # Get original image path.
    original_path = Path(row["image_path"])
    
    # Skip if original image does not exist.
    if not original_path.exists():
        continue
    
    # Normalize district and ward labels.
    district = str(row["district"]).lower().replace("district ", "")
    ward = str(row["ward"]).lower().replace(" ", "_")
    
    # Create output directory for segmented masks.
    out_dir = IMAGE_DIR / f"district_{district}" / ward / "segmented"
    out_dir.mkdir(parents = True, exist_ok = True)
    
    # Output path with class_ prefix.
    mask_path = out_dir / f"class_{uid}.png"
    
    # Skip if mask already exists.
    if mask_path.exists():
        processed_uids.add(uid)
        continue
    
    # Run segmentation.
    try:
        mask = segment_image(str(original_path))
        
        if mask is None:
            continue
        
        # Save mask as PNG.
        cv2.imwrite(str(mask_path), mask.astype(np.uint8))
        
        # Track progress.
        processed_uids.add(uid)
        results.append({
            "uid": uid,
            "mask_path": str(mask_path)
        })
        
    except Exception as e:
        print(f"Error processing UID {uid}: {e}")
        continue
    
    # Save checkpoint periodically.
    if len(processed_uids) % batch_size == 0:
        with open(CHECKPOINT_FILE, "w") as f:
            json.dump({"processed_uids": list(processed_uids)}, f)
        print(f"Checkpoint saved at {len(processed_uids)} processed images.")

# Final checkpoint save.
with open(CHECKPOINT_FILE, "w") as f:
    json.dump({"processed_uids": list(processed_uids)}, f)

print(f"Segmentation complete. Processed {len(processed_uids)} images.")

## VERIFY OUTPUTS

In [ ]:
# Count segmented masks per ward.
mask_counts = {}

for district_dir in IMAGE_DIR.iterdir():
    if not district_dir.is_dir():
        continue
    
    for ward_dir in district_dir.iterdir():
        if not ward_dir.is_dir():
            continue
        
        seg_dir = ward_dir / "segmented"
        if seg_dir.exists():
            mask_count = len(list(seg_dir.glob("class_*.png")))
            key = f"{district_dir.name}/{ward_dir.name}"
            mask_counts[key] = mask_count

print("SEGMENTATION SUMMARY")
for location, count in sorted(mask_counts.items()):
    print(f"{location}: {count} masks")

print(f"Total masks: {sum(mask_counts.values())}")

## VISUALIZE SAMPLE RESULTS

In [ ]:
import matplotlib.pyplot as plt

# Get sample image and mask.
sample_row = df.iloc[0]
sample_uid = sample_row["uid"]
sample_district = str(sample_row["district"]).lower().replace("district ", "")
sample_ward = str(sample_row["ward"]).lower().replace(" ", "_")

original_path = Path(sample_row["image_path"])
mask_path = IMAGE_DIR / f"district_{sample_district}" / sample_ward / "segmented" / f"class_{sample_uid}.png"

if original_path.exists() and mask_path.exists():
    # Load images.
    original = Image.open(original_path)
    mask = Image.open(mask_path)
    
    # Plot side by side.
    fig, axes = plt.subplots(1, 2, figsize = (16, 6))
    
    axes[0].imshow(original)
    axes[0].set_title(f"Original: gsv_{sample_uid}.jpg")
    axes[0].axis("off")
    
    axes[1].imshow(mask, cmap = "tab20")
    axes[1].set_title(f"Segmented: class_{sample_uid}.png")
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()
else:
    print("Sample files not found.")